# How is the sequence identity calculated by edlib ???

In [14]:
import os
import sys

import pandas as pd
import edlib

DATA_DIR="nfs/scratch/pdb_dimers"

In [15]:
def cd_hit_identity_edlib(seq1: str, seq2: str):
    """
    CD-HIT–style identity via edlib:
      - mode="HW": alignment must cover the whole query (the shorter sequence),
        but can start/end anywhere on the target (the longer sequence).
      - Count identical aligned columns (excluding gaps) and divide by len(shorter).
      - Return (identity, distance).
    """

    # Ensure query is the shorter sequence
    if len(seq1) <= len(seq2):
        query, target = seq1, seq2
    else:
        query, target = seq2, seq1

    # Align: path required so we can reconstruct matches
    res = edlib.align(query, target, mode="HW", task="path")

    # Reconstruct aligned strings
    nice = edlib.getNiceAlignment(res, query, target)
    print(f"Aligned Query:  {nice['query_aligned']}")
    print(f"Aligned Target: {nice['target_aligned']}")
    q_aln = nice["query_aligned"]
    t_aln = nice["target_aligned"]

    # Count exact matches in aligned columns
    matches = 0
    for qc, tc in zip(q_aln, t_aln):
        if qc != "-" and tc != "-" and qc == tc:
            matches += 1

    identity = matches / len(query)

    return identity

In [16]:
fix_1_100 = "AAAAAAGYAVHDHFYDAVVVGAGGAGLRAASGLVAHGLKTACISKVFPTRSHTVAAQGGINAALGNMTEDDWRWHAYDTVKGSDWLGDQDAIEHMCRLAPQVVLELESYGLPFSRTKEGKIYQRAFGGQSLKFGKGGQAYRCAAAADRTGHAILHTLYGMSLKYDCLFFIEYFALDLIMDQDGSCKGVIAMSMEDGSIHRFGAHQTVIATGGYGRAYQSCTSAHTCTGDGGGMVSRAGLPLQDLEFVQFHPTGIFPAGCLMTEGCRGEGGILRNSEGEPFMARYAPTAKDLASRDVVSRAMTLEIREGRGVGPNKDHIYLHLDHLPAETLRERLPGISETAKIFAGVDVTKEPIPVLPTVHYNMGGIPTNWKAQCLNPTSSDPNKIVPGLLAAGEAGSASVHGANRLGANSLLDLVVFGRTAADTVAEIVKPNSPPVTLPKDAGGGTIDRFDKIRHAKGPVSTADLRSKLQRTMQTRAPVYRNGDDLKKGCEEVREIMKEYKDVGIKDRTLVWNTDLIETLELENLITQAAQTIVSGEARKESRGAHAREDFTERDDKKWMKHSLSYQTKPHVEESDIVLKYRPVVDQPLDSEMHHVPPAKRVY"
fix_19_100 = "AACVRLYGPNFILQVYSSQRKSWHPVCQDDWNENYGRAACRDMGYKNNFYSSQGIVDDSGSTSFMKLNTSAGNVDIYKKLYHSDACSSKAVVSLRCIACGVNLNDDDDDK"

# calculate sequence identity using edlib
result = edlib.align(fix_1_100, fix_19_100, mode="NW", task="path")
edits = result["editDistance"]  # total edits is edit distance

print(f"Matches: {edits}, Len fix_1_100: {len(fix_1_100)}, Len fix_19_100: {len(fix_19_100)}")

Matches: 524, Len fix_1_100: 604, Len fix_19_100: 110


In [18]:
identity = 1 - edits / max(len(fix_1_100), len(fix_19_100))
print(f"Sequence Identity: {identity:.2%}")

Sequence Identity: 13.25%


In [19]:
local_identity = cd_hit_identity_edlib(fix_1_100, fix_19_100)
print(f"Local Identity (CD-HIT style): {local_identity:.2%}")

Aligned Query:  AACVRLYGPNFILQVYSSQRKSWHPVCQDDWNENYGRAACRDMGYKNNFYSSQGIVDDSGSTSFMKLNTSAGNVDIYKKLYHSDACSSKAVVSLRCIACGVNLNDDDDDK
Aligned Target: AAGEA--G--------SA---SVH-----------G-AN-RL-GA-N---SLLDLVVF-GRTAAD---TVAEIV---KPN--SPP-----VT-LPKDAGGGTI-DRFD-K
Local Identity (CD-HIT style): 24.55%


In [ ]:
cat = "cat"
code = "code"

# calculate sequence identity using edlib
result = edlib.align(cat, code, mode="NW", task="path")
edits = result["editDistance"]  # total edits is edit distance

print(f"Matches: {edits}, Len cat: {len(cat)}, Len code: {len(code)}")

Matches: 3, Len cat: 3, Len code: 4
